In [5]:
import math
import random
import copy
import networkx as nx
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict


random.seed(42)
np.random.seed(42)



Import del dataset

In [6]:
print("\n[1/6] Importazione grafo")
G = nx.read_edgelist("Dataset/facebook_combined.txt")
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
tutti_i_gradi = [grado for nodo, grado in G.degree()]
grado_massimo = max(tutti_i_gradi)
grado_minimo = min(tutti_i_gradi)
print(f"      Nodi: {n_nodes}, Archi: {n_edges}")
print(f"      Grado medio: {2*n_edges/n_nodes:.2f}")
print(f"      Grado massimo: {grado_massimo}")
print(f"      Grado minimo: {grado_minimo}")
print(f"      Clustering medio: {nx.average_clustering(G):.4f}")



[1/6] Importazione grafo
      Nodi: 4039, Archi: 88234
      Grado medio: 43.69
      Grado massimo: 1045
      Grado minimo: 1
      Clustering medio: 0.6055


Visualizza grafo

In [8]:
"""Visualizza proprietà del grafo."""
out_path = "/Users/giuseppepiosorrentino/RetiSocialiProj/Results"
degrees = [d for _, d in G.degree()]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(degrees, bins=30, color="#2196F3", edgecolor="white", alpha=0.85)
axes[0].set_title("Distribuzione dei gradi", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Grado", fontsize=11)
axes[0].set_ylabel("Frequenza", fontsize=11)
axes[0].grid(True, alpha=0.3)

# Draw small version of the graph
pos = nx.spring_layout(G, seed=42, k=0.5)
nx.draw_networkx(G, pos=pos, ax=axes[1], node_size=20, node_color="#2196F3",
                   edge_color="#BBBBBB", with_labels=False, alpha=0.8, width=0.5)
axes[1].set_title("Visualizzazione della rete (Barabási-Albert)", fontsize=13, fontweight="bold")
axes[1].axis("off")

fig.suptitle(f"Rete: n={G.number_of_nodes()}, m={G.number_of_edges()}, "
             f"<k>={2*G.number_of_edges()/G.number_of_nodes():.1f}",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Salvato: {out_path}")

  Salvato: /Users/giuseppepiosorrentino/RetiSocialiProj/Results


Inizializzazione delle Funzioni di costo

In [9]:
import random
import math

def cost_random(G, low=1, high=5, seed=42):
    rng = random.Random(seed)
    return {v: rng.randint(low, high) for v in G.nodes()}

def cost_half_degree(G):
    return {v: max(1, math.ceil(G.degree(v) / 2)) for v in G.nodes()}

#VALUTIAMO, è OPZIONALE
def cost_custom(G):
    """Costo basato su betweenness centrality: 
    i nodi ponte tra comunità costano di più"""
    bet = nx.betweenness_centrality(G)
    return {v: max(1, round(1 + bet[v] * 10)) for v in G.nodes()}

# Istanzia le tre funzioni costo sul grafo G
c_random = cost_random(G)
c_half   = cost_half_degree(G)
#OPZIONALE
c_custom = cost_custom(G)

print("Esempio costi (primi 5 nodi):")
for v in list(G.nodes())[:5]:
    print(f"  nodo {v}: random={c_random[v]}, d/2={c_half[v]}, custom={c_custom[v]}")



Esempio costi (primi 5 nodi):
  nodo 0: random=1, d/2=174, custom=2
  nodo 1: random=1, d/2=9, custom=1
  nodo 2: random=3, d/2=5, custom=1
  nodo 3: random=2, d/2=9, custom=1
  nodo 4: random=2, d/2=5, custom=1


In [21]:
costo_totale_random = sum(c_random.values())
costo_totale_half   = sum(c_half.values())
costo_totale_custom = sum(c_custom.values())
k_random = int(0.10 * costo_totale_random)
k_half   = int(0.10 * costo_totale_half)
k_custom = int(0.10 * costo_totale_custom)

print(f"Costo totale random: {costo_totale_random}")
print(f"Costo totale half:   {costo_totale_half}")
print(f"Costo totale custom: {costo_totale_custom}")

Costo totale random: 12173
Costo totale half:   89243
Costo totale custom: 4059


Introduciamo delle approssimazioni analitiche che stimano quello che la funzione obiettivo dobrebbe fare, simulare l'intera cascata per ogni nodo è costoso visto l'elevato numero di nodi nell'grafo. 
Le approssimazioni ci permettono  di stimare quanto S è efficace guardando solo la struttura locale del grafo (quanti vicini di ogni nodo sono già in S rispetto alla soglia).


In [10]:
from functools import lru_cache
import numpy as np

# Pre-calcola strutture del grafo una volta sola
def precompute_graph(G):
    neighbors = {v: set(G.neighbors(v)) for v in G.nodes()}
    degrees   = {v: G.degree(v) for v in G.nodes()}
    thresholds = {v: math.ceil(degrees[v] / 2) for v in G.nodes()}
    return neighbors, degrees, thresholds

neighbors, degrees, thresholds = precompute_graph(G)

def f1(G, S):
    S_set = frozenset(S)
    val = 0
    for v in G.nodes():
        overlap = len(neighbors[v] & S_set)
        val += min(overlap, thresholds[v])
    return val

def f2(G, S):
    S_set = frozenset(S)
    val = 0
    for v in G.nodes():
        overlap = len(neighbors[v] & S_set)
        t = thresholds[v]
        # somma aritmetica invece del ciclo: t + (t-1) + ... 
        if overlap > 0:
            effective = min(overlap, t)
            val += effective * t - (effective * (effective - 1)) // 2
    return val

def f3(G, S):
    S_set = frozenset(S)
    val = 0
    for v in G.nodes():
        dv = degrees[v]
        if dv == 0:
            continue
        overlap = len(neighbors[v] & S_set)
        t = thresholds[v]
        for i in range(1, overlap + 1):
            denom = dv - i + 1
            if denom > 0:
                val += max((t - i + 1) / denom, 0)
    return val

Algoritmo di Majority Cascade

In [11]:
def majority_cascade(G, S):
    """
    Simula il processo di attivazione con regola Majority.
    Un nodo v si attiva se almeno metà dei suoi vicini sono già attivi.
    Restituisce Inf[S] = insieme di tutti i nodi attivati.
    """
    infected = set(S)
    
    while True:
        new_infected = set()
        for v in G.nodes():
            if v not in infected:
                dv = G.degree(v)
                if dv == 0:
                    continue
                neighbors_infected = sum(1 for u in G.neighbors(v) if u in infected)
                if neighbors_infected >= math.ceil(dv / 2):
                    new_infected.add(v)
        
        if not new_infected:  # nessun nuovo nodo attivato: la cascata si ferma
            break
        infected |= new_infected
    
    return infected

In [12]:
# Prova con un piccolo seed set manuale
S_test = set(list(G.nodes())[:5])  # primi 5 nodi come seed
inf_test = majority_cascade(G, S_test)

print(f"Seed set: {S_test}")
print(f"Nodi attivati: {len(inf_test)} / {G.number_of_nodes()}")

Seed set: {'2', '3', '0', '4', '1'}
Nodi attivati: 53 / 4039


ALGORITMO 1: Cost-Seeds-Greedy

 invece di ricalcolare f su tutti i nodi, guarda solo i vicini del nodo appena aggiunto.

In [17]:
from functools import lru_cache

def marginal_gain(G, v, S_current, fi_func):
    """
    Ottimizzazione: invece di ricalcolare fi su tutto il grafo,
    guarda solo i vicini di v che cambiano.
    """
    S_set = frozenset(S_current)
    val = 0
    # Solo i vicini di v sono influenzati dall'aggiunta di v
    for u in neighbors[v]:
        overlap_before = len(neighbors[u] & S_set)
        overlap_after  = overlap_before + (1 if v not in S_set else 0)
        
        if fi_func == f1:
            val += min(overlap_after, thresholds[u]) - min(overlap_before, thresholds[u])
        elif fi_func == f2:
            t = thresholds[u]
            def contrib(ov):
                effective = min(ov, t)
                return effective * t - (effective * (effective - 1)) // 2
            val += contrib(overlap_after) - contrib(overlap_before)
        elif fi_func == f3:
            dv = degrees[u]
            if dv == 0:
                continue
            t = thresholds[u]
            before = sum(max((t-i+1)/( dv-i+1), 0) for i in range(1, overlap_before+1))
            after  = sum(max((t-i+1)/(dv-i+1), 0)  for i in range(1, overlap_after+1))
            val += after - before
    return val


invece di rivalutare tutti i nodi ad ogni iterazione, aggiorna solo quelli il cui vicinato è cambiato

In [15]:
def cost_seeds_greedy(G, k, c, fi_func, verbose=False):
    S_p = set()
    S_d = set()
    iteration = 0

    # Cache dei ratio: ricalcola solo i nodi il cui vicinato è cambiato
    ratio_cache = {v: fi_func(G, {v}) / c[v] for v in G.nodes() if c[v] > 0}

    while True:
        iteration += 1
        if not ratio_cache:
            break

        best_u = max(ratio_cache, key=ratio_cache.get)
        S_p = set(S_d)
        S_d = S_d | {best_u}
        del ratio_cache[best_u]

        if sum(c[u] for u in S_d) > k:
            break

        # Aggiorna solo i nodi vicini di best_u
        for v in neighbors[best_u]:
            if v not in S_d and c.get(v, 0) > 0:
                ratio_cache[v] = marginal_gain(G, v, S_d, fi_func) / c[v]

        if verbose:
            print(f"  iter {iteration}: nodo {best_u}, costo={sum(c[u] for u in S_d)}/{k}")

    return S_p

In [26]:
S_greedy_f1 = cost_seeds_greedy(G, k_random, c_random, f1)
print(f"Seed set Greedy-f1: {len(S_greedy_f1)} nodi, costo={sum(c_random[u] for u in S_greedy_f1)}")
print(f"\n|Inf[G, S_f1]| = {len(majority_cascade(G, S_greedy_f1))}")

Seed set Greedy-f1: 750 nodi, costo=1217

|Inf[G, S_f1]| = 2826


In [27]:
S_greedy_f2 = cost_seeds_greedy(G, k_random, c_random, f2)
#S_greedy_f3 = cost_seeds_greedy(G, k, c_random, f3)
print(f"Seed set Greedy-f2: {len(S_greedy_f2)} nodi, costo={sum(c_random[u] for u in S_greedy_f2)}")
#print(f"Seed set Greedy-f3: {len(S_greedy_f3)} nodi, costo={sum(c_random[u] for u in S_greedy_f3)}")
print(f"|Inf[G, S_f2]| = {len(majority_cascade(G, S_greedy_f2))}")
#print(f"|Inf[G, S_f3]| = {len(majority_cascade(G, S_greedy_f3))}")

Seed set Greedy-f2: 856 nodi, costo=1217
|Inf[G, S_f2]| = 2803


ALGORITMO 2: WTSS

In [2]:
def wtss_maximal(G, k, c):
    """
    WTSS adattato per restituire il seed set massimale con costo <= k.
    Threshold t(v) = ceil(d(v)/2) per ogni nodo v.
    
    Logica:
    - Case 1: v ha threshold 0 → viene attivato dai vicini, non serve nel seed
    - Case 2: v non può essere attivato dai vicini rimasti → va nel seed
    - Case 3: tutti attivabili → rimuovi il nodo "meno urgente" dal grafo
    """
    # Inizializzazione
    delta     = {v: degrees[v] for v in G.nodes()}        # grado corrente
    k_thresh  = {v: thresholds[v] for v in G.nodes()}     # threshold corrente
    neigh     = {v: set(neighbors[v]) for v in G.nodes()} # vicini correnti
    U         = set(G.nodes())
    S         = set()
    current_cost = 0

    while U:
        # Case 1: esiste v con threshold == 0 → attivato dai vicini
        case1 = next((v for v in U if k_thresh[v] == 0), None)
        if case1 is not None:
            v = case1
            for u in neigh[v]:
                if u in U:
                    k_thresh[u] = max(0, k_thresh[u] - 1)
                    delta[u] -= 1
                    neigh[u].discard(v)
            neigh[v] = set()
            U.remove(v)
            continue

        # Case 2: delta(v) < k(v) → non può essere attivato, va nel seed
        case2 = next((v for v in U if delta[v] < k_thresh[v]), None)
        if case2 is not None:
            v = case2
            if current_cost + c[v] > k:
                return S  # budget esaurito
            S.add(v)
            current_cost += c[v]
            for u in neigh[v]:
                if u in U:
                    k_thresh[u] = max(0, k_thresh[u] - 1)
                    delta[u] -= 1
                    neigh[u].discard(v)
            neigh[v] = set()
            U.remove(v)
            continue

        # Case 3: tutti attivabili → rimuovi il nodo meno urgente
        v = max(U, key=lambda u: (c[u] * k_thresh[u]) / (delta[u] * (delta[u] + 1)) 
                if delta[u] > 0 else float('inf'))
        for u in neigh[v]:
            if u in U:
                delta[u] -= 1
                neigh[u].discard(v)
        neigh[v] = set()
        U.remove(v)

    return S

In [29]:
S_wtss = wtss_maximal(G, k_random, c_random)
inf_wtss = majority_cascade(G, S_wtss)

print(f"Seed set WTSS: {len(S_wtss)} nodi, costo={sum(c_random[u] for u in S_wtss)}")
print(f"|Inf[G, S_wtss]| = {len(inf_wtss)} / {G.number_of_nodes()}")

Seed set WTSS: 768 nodi, costo=1217
|Inf[G, S_wtss]| = 3483 / 4039


ALGORITMO 3: Bridge-Density Seeding — algoritmo custom per la selezione del seed set.
    
Idea: un buon seed non è solo un nodo molto connesso, ma un nodo che può raggiungere zone "nuove" della rete non ancora coperte da altri seed.
Il punteggio di ogni nodo combina tre fattori:

1. d(v) = quanti vicini diretti può attivare
2. avg_neighbor_degree(v) = quanto sono influenti i suoi vicini (effetto moltiplicatore a 2 hop)
3. penalità di sovrapposizione → se i vicini di v sono già vicini di un seed scelto, v vale meno (evita seed ridondanti)
    
Formula:
score(v) = (d(v) * avg_neighbor_degree(v)) / (overlap + 1) / c(v)


In [19]:
def my_seeds(G, k, c):
    """
    Bridge-Density Seeding v2.
    
    Intuizione corretta: per la majority cascade un nodo si attiva
    solo se METÀ dei suoi vicini sono già attivi. Quindi i seed devono
    essere abbastanza vicini tra loro da "collaborare".
    
    Il punteggio premia nodi che:
      1. Hanno alto grado (molti vicini da attivare)
      2. Hanno alto clustering (i vicini si conoscono tra loro →
         facile raggiungere la soglia maggioranza)
      3. Hanno vicini già parzialmente coperti dai seed scelti
         (PREMIO invece di penalità → aiuta a superare la soglia)
    """
    S = set()
    current_cost = 0
    covered = set()  # vicini già raggiunti da seed precedenti
    
    # Pre-calcola clustering locale una volta sola
    clustering = nx.clustering(G)
    
    while True:
        best_u = None
        best_score = -1
        
        for v in set(G.nodes()) - S:
            if current_cost + c[v] > k:
                continue
            
            dv = degrees[v]
            if dv == 0:
                continue
            
            # Fattore 1: grado → capacità di diffusione diretta
            degree_score = dv
            
            # Fattore 2: clustering → i vicini si attivano a vicenda,
            # facilitando il raggiungimento della soglia maggioranza
            cluster_score = clustering[v] + 0.1  # +0.1 evita score=0
            
            # Fattore 3: overlap PREMIATO → più vicini di v sono già
            # coperti da seed, più v è vicino a far scattare la cascade
            # sui nodi in comune
            overlap = len(neighbors[v] & covered)
            overlap_bonus = 1 + overlap / max(dv, 1)
            
            # Punteggio finale
            score = (degree_score * cluster_score * overlap_bonus) / c[v]
            
            if score > best_score:
                best_score = score
                best_u = v
        
        if best_u is None:
            break
        
        S.add(best_u)
        current_cost += c[best_u]
        covered |= neighbors[best_u]
    
    return S

In [33]:
S_myseeds = my_seeds(G, k_random, c_random)
inf_myseeds = majority_cascade(G, S_myseeds)

print(f"Seed set My-Seeds: {len(S_myseeds)} nodi, costo={sum(c_random[u] for u in S_myseeds)}")
print(f"|Inf[G, S_myseeds]| = {len(inf_myseeds)} / {G.number_of_nodes()}")

Seed set My-Seeds: 618 nodi, costo=1217
|Inf[G, S_myseeds]| = 1258 / 4039


INIZIAMO CON L'ESPERIMENTO VERO E PROPRIO

Step 1: esperimenti al variare di k usando i seed set trovati e misurando |Inf[G,S]|

Step 2: rimuovo x archi a caso cioè le amicizie che finiscono e riapplico majority cascade con lo stesso S di prima

Step 3: rimuovo y nodi a caso   cioè le  persone che lasciano il social e riapllico majority

In [13]:
# Aggiungi punti intermedi nel range interessante
percentages = [0.05, 0.06, 0.07, 0.08, 0.09, 
               0.10, 0.12, 0.15, 0.20, 0.25, 
               0.30, 0.40, 0.50]
budgets_random = [int(p * sum(c_random.values())) for p in percentages]

inf_wtss = []

for k in budgets_random:
    S   = wtss_maximal(G, k, c_random)
    inf = majority_cascade(G, S)
    inf_wtss.append(len(inf))
    print(f"k={k:5d} | seed={len(S):4d} | |Inf|={len(inf):4d} / {G.number_of_nodes()}")

k=  608 | seed= 392 | |Inf|=1463 / 4039
k=  730 | seed= 475 | |Inf|=1826 / 4039
k=  852 | seed= 546 | |Inf|=2296 / 4039
k=  973 | seed= 627 | |Inf|=2861 / 4039
k= 1095 | seed= 697 | |Inf|=3233 / 4039
k= 1217 | seed= 767 | |Inf|=3481 / 4039
k= 1460 | seed= 892 | |Inf|=4039 / 4039
k= 1825 | seed= 892 | |Inf|=4039 / 4039
k= 2434 | seed= 892 | |Inf|=4039 / 4039
k= 3043 | seed= 892 | |Inf|=4039 / 4039
k= 3651 | seed= 892 | |Inf|=4039 / 4039
k= 4869 | seed= 892 | |Inf|=4039 / 4039
k= 6086 | seed= 892 | |Inf|=4039 / 4039


In [ ]:

budgets_used = [608, 730, 852, 973, 1095, 1217, 1460, 1825, 2434, 3043, 3651, 4869, 6086]
inf_wtss     = [1453, 1827, 2299, 2864, 3193, 3483, 4039, 4039, 4039, 4039, 4039, 4039, 4039]
plt.figure(figsize=(9, 5))
plt.plot(budgets_used, inf_wtss, color="#2196F3", marker="o", linewidth=2, label="WTSS")
plt.axhline(y=G.number_of_nodes(), color="gray", linestyle="--", alpha=0.5, label="|V|=4039")
plt.axvline(x=1460, color="#E91E63", linestyle="--", alpha=0.5, label="Saturazione k=1460")
plt.xlabel("Budget k")
plt.ylabel("|Inf[G, S]|")
plt.title("WTSS — |Inf[G,S]| al variare di k (c_random)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("Results/wtss_vs_k.png", dpi=150, bbox_inches="tight")



/var/folders/8z/67y4_fv93g3dgx30jl7w8ss40000gn/T/ipykernel_10374/2317306431.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


ORA eseguiamo step 1 su Greedy

In [45]:
inf_greedy_f1 = []
inf_greedy_f2 = []
inf_greedy_f3 = []

for k in budgets_random:
    S_f1 = cost_seeds_greedy(G, k, c_random, f1)
    S_f2 = cost_seeds_greedy(G, k, c_random, f2)
    S_f3 = cost_seeds_greedy(G, k, c_random, f3)
    
    inf_greedy_f1.append(len(majority_cascade(G, S_f1)))
    inf_greedy_f2.append(len(majority_cascade(G, S_f2)))
    inf_greedy_f3.append(len(majority_cascade(G, S_f3)))
    
    print(f"k={k:5d} | f1={inf_greedy_f1[-1]:4d} | f2={inf_greedy_f2[-1]:4d} | f3={inf_greedy_f3[-1]:4d}")

k=  608 | f1=1052 | f2= 661 | f3= 677
k=  730 | f1=1321 | f2=1216 | f3= 777
k=  852 | f1=1390 | f2=1867 | f3=1740
k=  973 | f1=2195 | f2=2119 | f3=2356
k= 1095 | f1=2628 | f2=2258 | f3=3103
k= 1217 | f1=2826 | f2=2803 | f3=3323
k= 1460 | f1=3195 | f2=3186 | f3=3674
k= 1825 | f1=3614 | f2=3439 | f3=3826
k= 2434 | f1=3882 | f2=3818 | f3=3993
k= 3043 | f1=3990 | f2=3906 | f3=4026
k= 3651 | f1=4032 | f2=3944 | f3=4039
k= 4869 | f1=4039 | f2=4023 | f3=4039
k= 6086 | f1=4039 | f2=4039 | f3=4039


STEP 1 su myseed

In [47]:
inf_myseeds   = []
for k in budgets_random:
    S_m = my_seeds(G, k, c_random)
    inf_myseeds.append(len(majority_cascade(G, S_m)))
    print(f"My-Seeds  | k={k:5d} | |Inf|={inf_myseeds[-1]:4d} / {G.number_of_nodes()}")

My-Seeds  | k=  608 | |Inf|= 820 / 4039
My-Seeds  | k=  730 | |Inf|= 943 / 4039
My-Seeds  | k=  852 | |Inf|= 967 / 4039
My-Seeds  | k=  973 | |Inf|=1079 / 4039
My-Seeds  | k= 1095 | |Inf|=1132 / 4039
My-Seeds  | k= 1217 | |Inf|=1233 / 4039
My-Seeds  | k= 1460 | |Inf|=1355 / 4039
My-Seeds  | k= 1825 | |Inf|=1577 / 4039
My-Seeds  | k= 2434 | |Inf|=1989 / 4039
My-Seeds  | k= 3043 | |Inf|=2163 / 4039
My-Seeds  | k= 3651 | |Inf|=2454 / 4039
My-Seeds  | k= 4869 | |Inf|=2944 / 4039
My-Seeds  | k= 6086 | |Inf|=3289 / 4039


In [48]:
plt.figure(figsize=(9, 5))

plt.plot(budgets_random, inf_wtss,      color="#2196F3", marker="o", linewidth=2, label="WTSS")
plt.plot(budgets_random, inf_greedy_f1, color="#E91E63", marker="s", linewidth=2, label="Greedy-f1")
plt.plot(budgets_random, inf_greedy_f2, color="#FF9800", marker="D", linewidth=2, label="Greedy-f2")
plt.plot(budgets_random, inf_greedy_f3, color="#9C27B0", marker="^", linewidth=2, label="Greedy-f3")
plt.plot(budgets_random, inf_myseeds,   color="#4CAF50", marker="v", linewidth=2, label="My-Seeds")

plt.axhline(y=G.number_of_nodes(), color="gray", linestyle="--", alpha=0.5, label="|V|=4039")
plt.xlabel("Budget k")
plt.ylabel("|Inf[G, S]|")
plt.title("Step 1 — confronto algoritmi al variare di k (c_random)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("Results/comparison.png", dpi=150, bbox_inches="tight")


In [21]:
# Funzione per rimuovere x archi casuali da G
def remove_edges(G, x, seed=42):
    G2 = G.copy()
    edges = list(G2.edges())
    rng = random.Random(seed)
    to_remove = rng.sample(edges, min(x, len(edges)))
    G2.remove_edges_from(to_remove)
    return G2

In [22]:
# Seed set di riferimento calcolati con k=1217
k_ref = 1217
S_ref_wtss  = wtss_maximal(G, k_ref, c_random)
S_ref_f1    = cost_seeds_greedy(G, k_ref, c_random, f1)
S_ref_f2    = cost_seeds_greedy(G, k_ref, c_random, f2)
S_ref_f3    = cost_seeds_greedy(G, k_ref, c_random, f3)
S_ref_my    = my_seeds(G, k_ref, c_random)

# Range di archi da rimuovere
x_range = list(range(0, 20000, 1000))

inf_wtss_edges = []
inf_f1_edges   = []
inf_f2_edges   = []
inf_f3_edges   = []
inf_my_edges   = []

for x in x_range:
    G2 = remove_edges(G, x)
    inf_wtss_edges.append(len(majority_cascade(G2, S_ref_wtss)))
    inf_f1_edges.append(len(majority_cascade(G2, S_ref_f1)))
    inf_f2_edges.append(len(majority_cascade(G2, S_ref_f2)))
    inf_f3_edges.append(len(majority_cascade(G2, S_ref_f3)))
    inf_my_edges.append(len(majority_cascade(G2, S_ref_my)))
    print(f"x={x:5d} | WTSS={inf_wtss_edges[-1]:4d} | f1={inf_f1_edges[-1]:4d} | f2={inf_f2_edges[-1]:4d} | f3={inf_f3_edges[-1]:4d} | My={inf_my_edges[-1]:4d}")

x=    0 | WTSS=3481 | f1=2826 | f2=2803 | f3=3323 | My=1258
x= 1000 | WTSS=3481 | f1=2826 | f2=2805 | f3=3324 | My=1260
x= 2000 | WTSS=3480 | f1=2985 | f2=2950 | f3=3244 | My=1267
x= 3000 | WTSS=3480 | f1=2877 | f2=2949 | f3=3196 | My=1258
x= 4000 | WTSS=3480 | f1=2879 | f2=2949 | f3=3220 | My=1261
x= 5000 | WTSS=3479 | f1=2884 | f2=2948 | f3=3254 | My=1260
x= 6000 | WTSS=3478 | f1=2885 | f2=2947 | f3=3264 | My=1263
x= 7000 | WTSS=3477 | f1=2885 | f2=2949 | f3=3260 | My=1278
x= 8000 | WTSS=3474 | f1=2886 | f2=2955 | f3=3377 | My=1280
x= 9000 | WTSS=3475 | f1=2878 | f2=2962 | f3=3264 | My=1284
x=10000 | WTSS=3474 | f1=2890 | f2=2967 | f3=3289 | My=1285
x=11000 | WTSS=3477 | f1=2979 | f2=2969 | f3=3286 | My=1282
x=12000 | WTSS=3475 | f1=2967 | f2=2962 | f3=3278 | My=1286
x=13000 | WTSS=3469 | f1=2868 | f2=2949 | f3=3501 | My=1281
x=14000 | WTSS=3467 | f1=2955 | f2=2950 | f3=3465 | My=1281
x=15000 | WTSS=3468 | f1=2956 | f2=2953 | f3=3467 | My=1278
x=16000 | WTSS=3469 | f1=2962 | f2=2951 

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(x_range, inf_wtss_edges, color="#2196F3", marker="o", linewidth=2, label="WTSS")
plt.plot(x_range, inf_f1_edges,   color="#E91E63", marker="s", linewidth=2, label="Greedy-f1")
plt.plot(x_range, inf_f2_edges,   color="#FF9800", marker="D", linewidth=2, label="Greedy-f2")
plt.plot(x_range, inf_f3_edges,   color="#9C27B0", marker="^", linewidth=2, label="Greedy-f3")
plt.plot(x_range, inf_my_edges,   color="#4CAF50", marker="v", linewidth=2, label="My-Seeds")

# Linea di riferimento: |Inf| sul grafo originale senza rimozioni
plt.axhline(y=3481, color="#2196F3", linestyle="--", alpha=0.3)
plt.axhline(y=2826, color="#E91E63", linestyle="--", alpha=0.3)
plt.axhline(y=2803, color="#FF9800", linestyle="--", alpha=0.3)
plt.axhline(y=3323, color="#9C27B0", linestyle="--", alpha=0.3)
plt.axhline(y=1258, color="#4CAF50", linestyle="--", alpha=0.3)

plt.xlabel("Archi rimossi (x)")
plt.ylabel("|Inf[G', S]|")
plt.title("Step 2 — Effetto rimozione archi (c_random, k=1217)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("Results/edge_removal.png", dpi=150, bbox_inches="tight")


/var/folders/8z/67y4_fv93g3dgx30jl7w8ss40000gn/T/ipykernel_14209/1066650650.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


ELIMINAZIONE DI NODI

In [25]:
def remove_vertices(G, y, seed=42):
    G2 = G.copy()
    nodes = list(G2.nodes())
    rng = random.Random(seed)
    to_remove = rng.sample(nodes, min(y, len(nodes)))
    G2.remove_nodes_from(to_remove)
    return G2

In [26]:
# Stesso k di riferimento
y_range = list(range(0, 1000, 50))

inf_wtss_nodes = []
inf_f1_nodes   = []
inf_f2_nodes   = []
inf_f3_nodes   = []
inf_my_nodes   = []

for y in y_range:
    G2 = remove_vertices(G, y)
    # I seed rimossi insieme ai nodi vanno esclusi
    S2_wtss = S_ref_wtss & set(G2.nodes())
    S2_f1   = S_ref_f1   & set(G2.nodes())
    S2_f2   = S_ref_f2   & set(G2.nodes())
    S2_f3   = S_ref_f3   & set(G2.nodes())
    S2_my   = S_ref_my   & set(G2.nodes())
    
    inf_wtss_nodes.append(len(majority_cascade(G2, S2_wtss)))
    inf_f1_nodes.append(len(majority_cascade(G2, S2_f1)))
    inf_f2_nodes.append(len(majority_cascade(G2, S2_f2)))
    inf_f3_nodes.append(len(majority_cascade(G2, S2_f3)))
    inf_my_nodes.append(len(majority_cascade(G2, S2_my)))
    print(f"y={y:4d} | WTSS={inf_wtss_nodes[-1]:4d} | f1={inf_f1_nodes[-1]:4d} | f2={inf_f2_nodes[-1]:4d} | f3={inf_f3_nodes[-1]:4d} | My={inf_my_nodes[-1]:4d}")

y=   0 | WTSS=3481 | f1=2826 | f2=2803 | f3=3323 | My=1258
y=  50 | WTSS=3443 | f1=2934 | f2=2887 | f3=3245 | My=1237
y= 100 | WTSS=3393 | f1=2909 | f2=2864 | f3=3371 | My=1219
y= 150 | WTSS=3349 | f1=2833 | f2=2686 | f3=3370 | My=1209
y= 200 | WTSS=3305 | f1=2836 | f2=2652 | f3=3365 | My=1205
y= 250 | WTSS=3120 | f1=2566 | f2=2726 | f3=3218 | My=1188
y= 300 | WTSS=3025 | f1=2530 | f2=2689 | f3=3255 | My=1179
y= 350 | WTSS=2964 | f1=2493 | f2=2642 | f3=3199 | My=1180
y= 400 | WTSS=2924 | f1=2463 | f2=2607 | f3=3159 | My=1144
y= 450 | WTSS=3059 | f1=2419 | f2=2582 | f3=3067 | My=1127
y= 500 | WTSS=3033 | f1=2388 | f2=2552 | f3=3108 | My=1117
y= 550 | WTSS=2987 | f1=2358 | f2=2519 | f3=3030 | My=1112
y= 600 | WTSS=2774 | f1=2322 | f2=2482 | f3=3026 | My=1102
y= 650 | WTSS=2728 | f1=2308 | f2=2461 | f3=2894 | My=1094
y= 700 | WTSS=2737 | f1=2276 | f2=2425 | f3=2829 | My=1095
y= 750 | WTSS=2699 | f1=2251 | f2=2393 | f3=2887 | My=1077
y= 800 | WTSS=2634 | f1=2226 | f2=2435 | f3=2820 | My=12

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(y_range, inf_wtss_nodes, color="#2196F3", marker="o", linewidth=2, label="WTSS")
plt.plot(y_range, inf_f1_nodes,   color="#E91E63", marker="s", linewidth=2, label="Greedy-f1")
plt.plot(y_range, inf_f2_nodes,   color="#FF9800", marker="D", linewidth=2, label="Greedy-f2")
plt.plot(y_range, inf_f3_nodes,   color="#9C27B0", marker="^", linewidth=2, label="Greedy-f3")
plt.plot(y_range, inf_my_nodes,   color="#4CAF50", marker="v", linewidth=2, label="My-Seeds")

plt.axhline(y=3481, color="#2196F3", linestyle="--", alpha=0.3)
plt.axhline(y=2826, color="#E91E63", linestyle="--", alpha=0.3)
plt.axhline(y=2803, color="#FF9800", linestyle="--", alpha=0.3)
plt.axhline(y=3323, color="#9C27B0", linestyle="--", alpha=0.3)
plt.axhline(y=1258, color="#4CAF50", linestyle="--", alpha=0.3)

plt.xlabel("Vertici rimossi (y)")
plt.ylabel("|Inf[G', S]|")
plt.title("Step 3 — Effetto rimozione vertici (c_random, k=1217)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("Results/node_removal.png", dpi=150, bbox_inches="tight")
